In [ ]:
!pip install -q openai pandas tqdm

In [ ]:
import os
import json
import time
import pandas as pd
from tqdm import tqdm
from getpass import getpass
from google.colab import drive
from openai import AzureOpenAI

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# AZURE OPENAI CLIENT
# ============================================================

os.environ["AZURE_OPENAI_API_KEY"] = getpass("Paste AZURE_OPENAI_API_KEY: ").strip()
os.environ["ENDPOINT_URL"] = getpass("Paste ENDPOINT_URL: ").strip()
os.environ["API_VERSION"] = getpass("Paste API_VERSION: ").strip()

client = AzureOpenAI(
    azure_endpoint=os.getenv("ENDPOINT_URL"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version= os.getenv("API_VERSION"),
)

# IMPORTANT:
# In Azure, model= should usually be your DEPLOYMENT NAME,
# not necessarily the public model name.
AZURE_DEPLOYMENT_NAME = "gpt-5-mini"

In [ ]:
# ============================================================
# CONFIG
# ============================================================

INPUT_DIR = "data/intermediate/topic_clean_balanced_diverse_selected"
OUTPUT_DIR = "data/final_repositories"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TOPIC_FILES = {
    "BLM": "BLM_selected_400_balanced_diverse.csv",
    "guns": "guns_selected_400_balanced_diverse.csv",
    "same_sex_marriage": "same_sex_marriage_selected_400_balanced_diverse.csv",
}

TEXT_COL = "comment_text"

FINAL_PER_TOPIC = 300
FINAL_PER_STANCE = 150

BATCH_SIZE = 25

In [ ]:
def topic_description(topic_name):
    if topic_name == "BLM":
        return "Black Lives Matter"
    if topic_name == "guns":
        return "Gun control"
    if topic_name == "same_sex_marriage":
        return "Same-sex marriage legalization"
    return topic_name

In [ ]:
def build_gpt_refinement_prompt(topic_name, stance, batch_df, text_col):
    rows = []

    for _, row in batch_df.iterrows():
        text = str(row[text_col]) if pd.notna(row[text_col]) else ""
        text = text.replace("\n", " ").strip()

        rows.append({
            "id": int(row["_gpt_local_id"]),
            "stance_label": str(row["stance_label"]),
            "text": text[:1500],
        })

    return f"""
You are helping refine a research dataset of political social-media comments.

Topic:
{topic_description(topic_name)}

Current stance group:
{stance}

Goal:
Select the best comments for a final dataset that is diverse, representative, and useful for building opposite-opinion personas.

All comments in this batch are already relevant to the topic and already classified as {stance}.
Your task is not to change the stance label.
Your task is to score how useful each comment is for the final selected set.

Evaluate each comment using:

1. uniqueness_score:
1-5, where 5 means this comment expresses a distinct or rare viewpoint within the {stance} side.

2. coverage_score:
1-5, where 5 means this comment represents an important recurring viewpoint within the {stance} side.

3. redundancy_risk:
1-5, where 5 means this comment is likely repetitive or similar to many others.

4. clarity_score:
1-5, where 5 means the stance is clear and understandable.

5. persona_usefulness:
1-5, where 5 means this comment would be useful as background for a realistic opposite-opinion persona.

6. keep_score:
1-10, where 10 means this comment should strongly be kept in the final dataset.

Important rules:
Do not prefer polite or moderate comments just because they are safer.
Preserve extreme, emotional, or unpopular comments if they represent a distinct stance.
Do not penalize toxicity unless the comment is too unclear to be useful.
Focus on stance diversity and coverage within the {stance} side.

Return valid JSON only in this exact format:

{{
  "items": [
    {{
      "id": 0,
      "uniqueness_score": 4,
      "coverage_score": 5,
      "redundancy_risk": 2,
      "clarity_score": 5,
      "persona_usefulness": 5,
      "keep_score": 9,
      "short_reason": "brief reason"
    }}
  ]
}}

Comments:
{json.dumps(rows, ensure_ascii=False)}
"""

In [ ]:
def extract_json_from_text(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end + 1])
        raise

In [ ]:
def score_batch_with_gpt(topic_name, stance, batch_df, text_col, max_retries=4):
    prompt = build_gpt_refinement_prompt(topic_name, stance, batch_df, text_col)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=AZURE_DEPLOYMENT_NAME,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a careful research assistant. Return valid JSON only."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
            )

            content = response.choices[0].message.content
            parsed = extract_json_from_text(content)

            return parsed.get("items", [])

        except Exception as e:
            print(f"Attempt {attempt + 1} failed:", repr(e))
            time.sleep(5 + attempt * 5)

    return []

In [ ]:
def gpt_score_topic(topic_name, input_path):
    print("\n" + "=" * 80)
    print(f"GPT final scoring: {topic_name}")
    print("=" * 80)

    df = pd.read_csv(input_path, encoding="utf-8-sig")

    if TEXT_COL not in df.columns:
        raise ValueError(f"TEXT_COL '{TEXT_COL}' not found. Columns: {df.columns.tolist()}")

    if "stance_label" not in df.columns:
        raise ValueError("Missing stance_label column.")

    df["stance_label"] = df["stance_label"].astype(str).str.lower().str.strip()
    df = df[df["stance_label"].isin(["pro", "anti"])].copy()
    df = df.reset_index(drop=True)

    df["_gpt_local_id"] = range(len(df))

    all_scores = []

    for stance in ["pro", "anti"]:
        stance_df = df[df["stance_label"] == stance].copy()

        print(f"\nScoring {stance}: {len(stance_df)} rows")

        for start in tqdm(range(0, len(stance_df), BATCH_SIZE), desc=f"{topic_name} {stance}"):
            batch = stance_df.iloc[start:start + BATCH_SIZE]
            items = score_batch_with_gpt(topic_name, stance, batch, TEXT_COL)
            all_scores.extend(items)
            time.sleep(0.5)

    scores_df = pd.DataFrame(all_scores)

    if scores_df.empty:
        scores_df = pd.DataFrame(columns=[
            "id",
            "uniqueness_score",
            "coverage_score",
            "redundancy_risk",
            "clarity_score",
            "persona_usefulness",
            "keep_score",
            "short_reason",
        ])

    rename_map = {
        "reason": "short_reason",
        "rationale": "short_reason",
    }

    scores_df = scores_df.rename(columns={
        old: new for old, new in rename_map.items() if old in scores_df.columns
    })

    required_defaults = {
        "id": None,
        "uniqueness_score": 3,
        "coverage_score": 3,
        "redundancy_risk": 3,
        "clarity_score": 3,
        "persona_usefulness": 3,
        "keep_score": 5,
        "short_reason": "missing_or_parse_error",
    }

    for col, default in required_defaults.items():
        if col not in scores_df.columns:
            scores_df[col] = default

    scores_df["id"] = pd.to_numeric(scores_df["id"], errors="coerce")
    scores_df = scores_df.dropna(subset=["id"])
    scores_df["id"] = scores_df["id"].astype(int)

    numeric_cols = [
        "uniqueness_score",
        "coverage_score",
        "redundancy_risk",
        "clarity_score",
        "persona_usefulness",
        "keep_score",
    ]

    for col in numeric_cols:
        scores_df[col] = pd.to_numeric(scores_df[col], errors="coerce").fillna(required_defaults[col])

    df_clean = df.drop(
        columns=[
            "id",
            "uniqueness_score",
            "coverage_score",
            "redundancy_risk",
            "clarity_score",
            "persona_usefulness",
            "keep_score",
            "short_reason",
            "gpt_final_score",
        ],
        errors="ignore"
    )

    merged = df_clean.merge(
        scores_df[
            [
                "id",
                "uniqueness_score",
                "coverage_score",
                "redundancy_risk",
                "clarity_score",
                "persona_usefulness",
                "keep_score",
                "short_reason",
            ]
        ],
        left_on="_gpt_local_id",
        right_on="id",
        how="left"
    )

    for col in numeric_cols:
        merged[col] = pd.to_numeric(merged[col], errors="coerce").fillna(required_defaults[col])

    merged["short_reason"] = merged["short_reason"].fillna("missing_or_parse_error")

    merged["gpt_final_score"] = (
        merged["keep_score"]
        + 0.9 * merged["coverage_score"]
        + 0.9 * merged["uniqueness_score"]
        + 0.6 * merged["persona_usefulness"]
        + 0.4 * merged["clarity_score"]
        - 0.8 * merged["redundancy_risk"]
    )

    scores_path = os.path.join(OUTPUT_DIR, f"{topic_name}_gpt_scores_all_400.csv")
    merged.to_csv(scores_path, index=False, encoding="utf-8-sig")

    print("Saved all GPT scores:", scores_path)

    final_parts = []

    for stance in ["pro", "anti"]:
        stance_scored = merged[merged["stance_label"] == stance].copy()

        if len(stance_scored) < FINAL_PER_STANCE:
            raise ValueError(
                f"{topic_name} has only {len(stance_scored)} rows for stance '{stance}', "
                f"cannot select {FINAL_PER_STANCE}."
            )

        chosen = (
            stance_scored
            .sort_values("gpt_final_score", ascending=False)
            .head(FINAL_PER_STANCE)
            .copy()
        )

        chosen["gpt_final_stance_rank"] = range(1, len(chosen) + 1)
        final_parts.append(chosen)

    final_df = pd.concat(final_parts, ignore_index=True)

    final_df = final_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    final_df["topic_gpt_final"] = topic_name
    final_df["gpt_final_rank_overall"] = range(1, len(final_df) + 1)

    final_path = os.path.join(OUTPUT_DIR, f"{topic_name}_final_300_150pro_150anti_gpt.csv")
    final_df.to_csv(final_path, index=False, encoding="utf-8-sig")

    print("Saved final:", final_path)
    print(final_df["stance_label"].value_counts())

    return final_df


all_final = []

for topic_name, filename in TOPIC_FILES.items():
    input_path = os.path.join(INPUT_DIR, filename)

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Missing file: {input_path}")

    final_df = gpt_score_topic(topic_name, input_path)
    all_final.append(final_df)

combined = pd.concat(all_final, ignore_index=True)

combined_path = os.path.join(
    OUTPUT_DIR,
    f"all_topics_final_{len(combined)}_150pro_150anti_gpt.csv"
)

combined.to_csv(combined_path, index=False, encoding="utf-8-sig")

print("Saved combined:", combined_path)
print(combined.groupby(["topic_gpt_final", "stance_label"]).size())